In [1]:
##########################################
##########################################
##########################################


# promtps específicos por APP



In [15]:
prompt_onboarding_causa_raiz="""
#CONTEXT
You are a top-tier incident analyst focused on reducing the number of tickets.
You are contracted to analyze the main root cause of each incident and propose a concrete fix.
Each incident provides two fields:
- desc: problem description
- res: resolution notes for the ticket
- ticket_type: type of the ticket, request or incident.

#TASK
1) From one incident, identify the main problem(s) and propose a viable, actionable solution.
2) Detect these groups/conditions ONLY when explicitly supported by the text (desc or res):

   2.1) Flujo de onboarding bloqueado y tareas pendientes/borrador
   Use this group when the onboarding workflow is blocked or not progressing, including cases such as:
   - Previous tasks still active, unfinished, or in Draft/Ready/Pending status
   - HR dependency or pending approval preventing progression
   - User cannot continue because the process did not advance automatically
   - Portal flow is stopped due to pending internal workflow steps
   - A previous onboarding step is still open and blocks the next one

   2.2) Faltan dispositivos / activos no provisionados
   Use this group when there is explicit evidence such as:
   - Employee has not received laptop, mobile, token, or other devices
   - Devices/assets are missing, not registered, or not delivered
   - Lack of assets explicitly blocks onboarding or first-day access
   - Mobile/device cannot be used because the asset is not provisioned
   - User cannot start because required hardware has not been assigned

   2.3) Errores en datos de contacto y autenticación MFA
   Use this group when there is explicit evidence of issues such as:
   - Incorrect phone number
   - Incorrect email used for access or communications
   - MFA code/PIN not received or sent incorrectly
   - Authentication blocked by wrong contact data
   - Password/credentials delivery issue tied to contact/authentication data

   2.4) Gestión deficiente de licencias y cuentas de acceso (M365/AD)
   Use this group when the issue is explicitly related to:
   - Missing or inactive license
   - User/account not activated
   - Mailbox not created or not provisioned
   - Domain/AD/M365 account provisioning failed
   - Access blocked because account setup/provisioning is incomplete

   2.5) Fallas en sincronización e integración entre sistemas (SAP/ServiceNow/Weblayer)
   Use this group when the incident explicitly mentions:
   - Data not syncing between systems
   - Company, position, or process data not updated across tools
   - Integration failure between onboarding and external systems
   - Mismatched fields between systems
   - Process blocked due to system-to-system synchronization failure

   2.6) Clasificación incorrecta de peticiones y falta de autoayuda
   Use this group when the ticket is explicitly not a technical incident, for example:
   - The resolution says it is a request/petition rather than an incident
   - The ticket type is Petición
   - The user is only asking for status, manual activation, or information
   - There is no actual technical failure described
   - The request should have gone through another service/request channel

3) Classify the root cause with ONE label from “CLASSIFICATION OPTIONS”.
4) Output ONLY a JSON object with EXACTLY these 10 fields, in this exact order:
   classification, data_missing, personal_data, request_modification,
   workflow_stopped, previous_task_active, licence_inactive, upload_information,
   access_app_error, task_stopped

#OUTPUT SCHEMA (STRICT, EXACT ORDER)
Return ONLY a single-line JSON object with EXACTLY these 12 fields, in this exact order:
1. `classification`
2. `data_missing`
3. `personal_data`
4. `request_modification`
5. `workflow_stopped`
6. `previous_task_active`
7. `licence_inactive`
8. `upload_information`
9. `access_app_error`
10. `task_stopped`

#FIELD RULES (STRICT)
- Language: All JSON string values must be in Spanish.
- data_missing: array of attributes missing or invalid/incorrect ONLY if explicitly mentioned
  (e.g., `DNI`, `email`, `fecha de nacimiento`, `teléfono`, or other attributes explicitly mentioned).
  Use [] if none.
- personal_data, request_modification, workflow_stopped, previous_task_active, licence_inactive,
  upload_information, access_app_error: each must be `S`, `N` or `I` only.
  Use `S` ONLY if there is explicit evidence in desc or res; otherwise `N`.
  If the text is speculative (`posible`, `parece`, `podría`, `se sospecha`), treat as NO evidence → `I`.
- task_stopped: array with names of unfinished previous tasks EXACTLY as found in the text when the workflow is stopped.
  Extract names ONLY when they appear after the literal prefix `Task ` (example: `Task Verificación de Identidad`).
  If the pattern is not present, use [] (do not invent task names).

#DECISION RULES (EVIDENCE-BASED)
- Evidence over inference: mark `S` only with clear explicit evidence; otherwise `N`.
- If multiple classifications could apply, choose ONE single most specific using this priority:
  1) Errores en datos de contacto y autenticación MFA
  2) Gestión deficiente de licencias y cuentas de acceso (M365/AD)
  3) Fallas en sincronización e integración entre sistemas (SAP/ServiceNow/Weblayer)
  4) Faltan dispositivos / activos no provisionados
  5) Flujo de onboarding bloqueado y tareas pendientes/borrador
  6) Clasificación incorrecta de peticiones y falta de autoayuda
  7) Otros

- Unknown/insufficient info:
  If details are insufficient, set flags to "N", arrays to [], classification to "7. Otros — Cualquier otro caso no cubierto.",
  rootcause to "Información insuficiente", and propose a solution that requests the exact missing data.

#GROUP BOUNDARIES (CONSISTENCY)
- "Errores en datos de contacto y autenticación MFA" dominates when the explicit root issue is wrong phone/email/PIN/MFA/auth contact data.
- "Gestión deficiente de licencias y cuentas de acceso (M365/AD)" dominates when the user cannot proceed due to account, mailbox, domain, provisioning, or license setup.
- "Fallas en sincronización e integración entre sistemas (SAP/ServiceNow/Weblayer)" dominates when the issue is caused by fields, company, position, or process data not syncing between systems.
- "Faltan dispositivos / activos no provisionados" dominates when the explicit blocker is missing laptop, mobile, token, or other assigned assets.
- "Flujo de onboarding bloqueado y tareas pendientes/borrador" dominates when the process is stuck due to pending tasks, HR dependency, Draft/Ready states, or onboarding flow not moving forward, excluding cases where the explicit main blocker is missing devices.
- "Clasificación incorrecta de peticiones y falta de autoayuda" dominates when the text explicitly indicates there is no technical incident and the ticket is only a request or status inquiry.

#BOOLEAN FLAGS CONSISTENCY (POST-CHECKS)
Apply these consistency rules after classification and extraction:
- If classification is "1. Errores en datos de contacto y autenticación MFA." → personal_data="S".
- If classification is "2. Gestión deficiente de licencias y cuentas de acceso (M365/AD)." → licence_inactive="S" ONLY if the text explicitly mentions missing/inactive license; otherwise keep according to evidence.
- If classification is "3. Fallas en sincronización e integración entre sistemas (SAP/ServiceNow/Weblayer)." → workflow_stopped="S" ONLY if the sync failure explicitly blocks the process.
- If classification is "4. Faltan dispositivos / activos no provisionados." → workflow_stopped="S" ONLY if the text explicitly says the missing device blocks onboarding or access.
- If classification is "5. Flujo de onboarding bloqueado y tareas pendientes/borrador." → workflow_stopped="S".
- If classification is "6. Clasificación incorrecta de peticiones y falta de autoayuda." → keep all operational flags as found by evidence, defaulting to "N" if not explicit.
- If previous_task_active="S" OR licence_inactive="S" OR upload_information="S" → workflow_stopped="S".
- If task_stopped has one or more items → previous_task_active="S" AND workflow_stopped="S".
- workflow_stopped / previous_task_active / licence_inactive / upload_information:
  set according to explicit onboarding evidence only. Otherwise keep "N".

#CLASSIFICATION DETECTION RULES
1) Errores en datos de contacto y autenticación MFA
   Match when: explicit issue with phone, email, PIN, MFA, SMS, or credential delivery due to wrong contact/authentication data.
   Keywords: "teléfono incorrecto", "email incorrecto", "PIN incorrecto", "MFA", "SMS", "código no recibido", "número erróneo", "credenciales enviadas al contacto incorrecto".

2) Gestión deficiente de licencias y cuentas de acceso (M365/AD)
   Match when: explicit issue with license, mailbox, account activation, domain user, M365, AD, or incomplete account provisioning.
   Keywords: "licencia", "licence", "mailbox", "buzón", "M365", "Active Directory", "AD", "dominio", "cuenta no activada", "usuario no provisionado".

3) Fallas en sincronización e integración entre sistemas (SAP/ServiceNow/Weblayer)
   Match when: explicit issue with sync/integration between tools or mismatched fields across systems.
   Keywords: "no sincroniza", "sync", "ServiceNow", "SNOW", "SAP", "Weblayer", "posición", "empresa", "campo no actualizado", "integración", "desalineado".

4) Faltan dispositivos / activos no provisionados
   Match when: explicit issue with missing laptop, mobile, token, or any required hardware/asset not yet assigned, registered, delivered, or usable.
   Keywords: "sin portátil", "sin laptop", "sin móvil", "sin dispositivo", "faltan dispositivos", "activo no registrado", "asset no provisionado", "no entregado", "no asignado".

5) Flujo de onboarding bloqueado y tareas pendientes/borrador
   Match when: the process is explicitly stopped or not moving because tasks remain open/pending/draft or workflow dependencies are unresolved.
   Keywords: "workflow bloqueado", "no avanza", "Draft", "Ready", "Pending", "tarea pendiente", "RRHH", "aprobación pendiente", "onboarding incompleto", "task activa".

6) Clasificación incorrecta de peticiones y falta de autoayuda
   Match when: explicit indication that the ticket is a request, inquiry, or non-incident.
   Keywords: "petición", "no es una incidencia", "consulta", "solicitud", "estado", "activar manualmente", "información".

7) Otros — Cualquier caso no cubierto
   Use only when none of the above match with explicit evidence.

#CLASSIFICATION OPTIONS (exact strings; do not translate or alter)
1. Errores en datos de contacto y autenticación MFA.
2. Gestión deficiente de licencias y cuentas de acceso (M365/AD).
3. Fallas en sincronización e integración entre sistemas (SAP/ServiceNow/Weblayer).
4. Faltan dispositivos / activos no provisionados.
5. Flujo de onboarding bloqueado y tareas pendientes/borrador.
6. Clasificación incorrecta de peticiones y falta de autoayuda.
7. Otros — Cualquier otro caso no cubierto.

#QUALITY CHECK (BEFORE FINALIZING)
- Output must be ONLY a single-line JSON object (no markdown, no extra text).
- Ensure the 10 fields are present and in the exact order.
- Ensure all JSON string values are in Spanish.
- Ensure only "S", "N", "I" are used for boolean fields.
- Ensure arrays use [] when empty and contain exact extracted task names only.

#INPUT
The incident description is: {desc}
The resolution notes are: {res}
The ticket type is: {ticket_type}

#FINAL OUTPUT FORMAT (return only this JSON; replace values accordingly)
{{"classification":"...","data_missing":[],"personal_data":"N","request_modification":"N","workflow_stopped":"N","previous_task_active":"N","licence_inactive":"N","upload_information":"N","access_app_error":"N","task_stopped":[]}}
"""


pp=""" 
    #CONTEXT
    You are a top-tier incident analyst focused on reducing the number of tickets. 
    You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
    Each incident provides two fields: desc: problem description, es: resolution notes for the ticket.

    #TASK
     1. From one incident, identify the main problem(s) and propose a viable, actionable solution.
     2. Detect these groups/conditions ONLY when explicitly supported by the text:
     2.1. Group 1 (personal data missing): DNI, birthdate, email, telephone number.
     2.2. Group 2 (request to change process data): e.g., start/end dates, email, etc.
     2.3. Group 3 (onboarding workflow not progressing) due to any of:
     - User not logged ina) Tasks not finished (e.g., Draft, Ready, etc.)
     - Missing/Inactive license
     - User not active in the Enelint domain
     - User cannot upload information to the portal
     2.4. Group 4 (no access) due lacks access to the portal or to another application.
     3. Classify the root cause with ONE label from “CLASSIFICATION OPTIONS”.
     4. Output ONLY a JSON object with EXACTLY these 12 fields, in this exact order: classification, rootcause, solution, data_missing, personal_data, request_modification, workflow_stopped, previous_task_active, licence_inactive, upload_information, access_app_error, task_stopped

     #FIELD RULES (STRICT)
     - Language: All JSON string values must be in Spanish.
     - rootcause: max 10 words; concise summary of the cause.
     - solution: specific, practical action the analyst would take or request.
     - data_missing: array of attributes missing or incorrect (e.g., "DNI", "email", "fecha de nacimiento", "teléfono", or other attributes explicitly mentioned). Use [] if none.
     - personal_data, request_modification, workflow_stopped, previous_task_active, licence_inactive, upload_information, access_app_error: each must be "S" or "N" only. Use "S" ONLY if there is explicit evidence in desc or res; otherwise "N".
     - task_stopped: array with names of unfinished previous tasks EXACTLY as found in the text when the workflow is stopped. Extract names that appear after the literal prefix "Task " (example: "Task Verificación de Identidad"). If multiple, include all. Use [] if none.

     #DECISION RULES
     - Evidence over inference: mark "S" only with clear evidence; otherwise "N".
     - If multiple classifications could apply, choose the single most specific using this priority:
     1) Grupo 1
     2) Grupo 2
     3) Grupo 3
     4) Grupo 4
     5) Faltan datos
     6) Datos incorrectos
     7) Problemas VPN
     8) Problemas Teams
     9) Problemas de conexión
     10) Problemas de permisos
     11) Problemas de software12) Problemas de hardware
     13) Problemas de configuración
     14) Otros
     - Unknown/insufficient info: if details are insufficient, set flags to "N", arrays to [], classification to "Otros", rootcause to "Información insuficiente", and propose a solution that requests the exact missing data.

    #CLASSIFICATION DETECTION RULES (for 5–14)
    5) Faltan datos — Problemas por datos faltantes (no personales)
       Match when: required non-personal info is missing (documents, case IDs, attachments, forms).
       Keywords: "falta", "no adjunta", "no aporta", "incompleto", "sin documento", "missing attachment", "not provided", "no indica", "campo vacío".
       Exclude: personal data → that is Grupo 1.
    
    6) Datos incorrectos — Problemas por datos erróneos
       Match when: data present but wrong/invalid/inconsistent.
       Keywords: "erróneo", "incorrecto", "equivocado", "inconsistente", "formato inválido", "mal cargado", "mismatch", "typo".
       Examples: email mal escrito, fecha inválida, número fuera de rango.

    7) Problemas VPN — Incidencias por la VPN
       Match when: failure tied to VPN usage.
       Keywords/brands: "VPN", "GlobalProtect", "AnyConnect", "FortiClient", "túnel", "split tunnel", "cuando conecto la VPN", "se corta la VPN", "no levanta túnel".
       Prefer over generic connection issues (priority 7 > 9).
    
    8) Problemas Teams — Microsoft Teams
       Match when: issue mentions Microsoft Teams (app, calls, chat, meetings).
       Keywords: "Teams", "Microsoft Teams", "no inicia Teams", "no puedo unirme", "no sincroniza", "no recibe llamadas", "grabación Teams".
    
    9) Problemas de conexión — Red (no VPN)
       Match when: connectivity/latency/DNS/proxy/timeouts not tied to VPN.
       Keywords: "sin internet", "caída de red", "latencia", "no resuelve DNS", "timeout", "proxy", "no carga página", "packet loss", "ping falla".
       Exclude if VPN is explicitly involved → use Problemas VPN.
    
    10) Problemas de permisos — Roles/visibilidad/autorización dentro de apps
        Match when: user enters the app but lacks rights/visibility or sees permission errors.
        Keywords: "sin permisos", "permiso denegado", "no le aparece", "no ve datos", "rol", "perfil", "RBAC", "autorización", "alta de permisos", "asignar rol".
        Heuristic: Whole-app/portal access blocked → Grupo 4. Access ok but missing rights → Problemas de permisos.
    
    11) Problemas de software — Fallos genéricos de software
        Match when: application bug/crash/installation/compatibility not specific to Teams or configuration.
        Keywords: "bug", "error de aplicación", "se cierra", "crashea", "exception", "stacktrace", "no instala", "incompatible", "actualización fallida".
        Exclude: if clearly config file/parameter issue → Problemas de configuración (13).
    
    12) Problemas de hardware — Dispositivo físico
        Match when: physical device failure.
        Keywords: "teclado", "ratón", "monitor", "impresora", "portátil", "batería", "no enciende", "dañado", "golpe", "sobrecalentamiento", "LED parpadea", "ruido disco".
    
    13) Problemas de configuración — Parámetros/archivos/policies
        Match when: wrong config, file, path, environment variable, registry, policy, DNS/proxy settings explicitly misconfigured.
        Keywords: "configuración", "parámetro", "ruta", "variable de entorno", "registro", "policy", "GPO", ".ini", ".conf", "yml/json", "proxy mal configurado", "DNS mal configurado".
        Prefer this over generic software if config is the root cause.
    
    14) Otros — Cualquier caso no cubierto
        Use only when none of the above (or Grupo 1–4) match with evidence.
    
    #BOOLEAN FLAGS CONSISTENCY (mapping hints)
    - access_app_error = "S" when there is inability to access a system/app (Grupo 4) OR explicit permission/auth errors (Problemas de permisos). Otherwise "N".
    - workflow_stopped / previous_task_active / licence_inactive / upload_information: set according to onboarding evidence only (Grupo 3 subcauses).
    - personal_data = "S" only for missing/invalid personal data (Grupo 1).
    - request_modification = "S" when the user requests changing process data (Grupo 2).
    - data_missing: list only attributes explicitly missing/incorrect (non-personal go to this list; personal data also list them here when Grupo 1 applies).
    
    #CLASSIFICATION OPTIONS (exact strings; do not translate or alter)
    1. Grupo 1 — Faltan datos personales (DNI, fecha de nacimiento, email, teléfono).
    2. Grupo 2 — Solicitud de cambio de datos del proceso (fechas de inicio/fin, email, etc.).
    3. Grupo 3 — El workflow de onboarding no progresa.
    4. Grupo 4 — Sin acceso al portal u otra aplicación.
    5. Faltan datos — Problemas por datos faltantes (en general).
    6. Datos incorrectos — Problemas por datos erróneos.
    7. Problemas VPN — Incidencias de conexión debidas a la VPN.
    8. Problemas Teams — Incidencias con la aplicación Microsoft Teams.
    9. Problemas de conexión — Incidencias de red no relacionadas con la VPN.
    10. Problemas de permisos — Falta de permisos, acceso o visibilidad en aplicaciones/sistemas.
    11. Problemas de software — Fallos genéricos de software.
    12. Problemas de hardware — Fallos de dispositivos físicos.
    13. Problemas de configuración — Errores en archivos o parámetros de configuración.
    14. Otros — Cualquier otro caso no cubierto.

    #QUALITY CHECK (before finalizing)
    - Ensure all strings are in Spanish.
    - Ensure the 12 fields are present and in the exact order.
    - Ensure only "S" or "N" are used for the specified boolean fields.
    - Ensure rootcause has 10 words or fewer.
    - Ensure arrays use [] when empty and contain exact task names after "Task ".

    #INPUT
    The incident description is: {desc}
    The resolution notes are: {res}
    
    #FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
    {{
        "classification": "...","rootcause": "...","solution": "...","data_missing": [],"personal_data": "N",
        "request_modification": "N","workflow_stopped": "N","previous_task_active": "N","licence_inactive": "N",
        "upload_information": "N","access_app_error": "N","task_stopped": []
    }}
"""


In [16]:
prompt_hranalytics_causa_raiz=""" 
#CONTEXT
    You are a top-tier incident analyst focused on reducing the number of tickets. 
    You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
    Each incident provides two fields: desc: problem description, res: resolution notes for the ticket.

#TASK
    1. From one incident, identify the main problem(s) and propose a viable, actionable solution.
    2. Detect these groups/conditions ONLY when explicitly supported by the text:
     2.1. Problem with disk space.
     2.2. Problem accesing a la platform.
     2.3. Problem with data format between source system and the configuration of the load configurated.
     2.4. Problem with configuration or roles.
     2.5. Problem with visualization of performance.
     2.6. Problem with the daily load planned.
     2.7. Other kind of problems

#FIELD RULES (STRICT)
- Language: All JSON string values must be in Spanish.
- Create three new fields: classification, rootcause and solution.

#CLASSIFICATION FIELD
- Create a new field called *classification* with the origen of the root cause.
- The classification field can only have the following values: Group1, Group2, Group3, Group4, Group5, Group6, Group7.
- If multiple classification could apply, choose the single most specific using the following value and order:
    -- Match the value Group1 when related with disk space. 
    -- Match the value Group2 when related with problem accesing at the platform. 
    -- Match the value Group3 when related with format problem in the interface.
    -- Match the value Group4 when related with roles or configuration. 
    -- Match the value Group5 when related with performance or visualization. 
    -- Match the value Group6 when related with planification or load processes. 
    -- Match the value Group7 when related with other problems.

#ROOTCAUSE FIELD
- Create a new field called *rootcause* with the short description of the root cause.
- The rootcause field should have a max of 10 words with a concise summary of the cause. 

#SOLUTION FIELD
- Create a new filed called *solution* with a short description of the solution for the incident. 
- The solution field should have a specific, practical action the analyst would take or request. 

#QUALITY CHECK (before finalizing)
- Ensure all new fields are written in Spanish.
- Ensure the 3 fields are present and in the exact order.
- Ensure rootcause has 10 words or fewer.

#INPUT
The incident description is: '''{desc}'''
The resolution notes are: '''{res}'''
    
#FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
{{"classification": "...","rootcause": "...","solution": "..."}}
"""

In [17]:
prompt_webuy_causa_raiz=""" 
#CONTEXT.
You are a top-tier incident analyst focused on reducing the number of tickets. 
You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
Each incident provides two fields: desc: problem description, res: resolution notes for the ticket.

#TASK.
1. From one incident, identify the main problem(s) and propose a viable, actionable solution.
2. Based on the incident description and resolution, detect which of the following groups best matches the main root cause. 
    Only assign a group when it is explicitly supported by the text:
    2.1. Not enough incident description or incident description is not provided.
    2.2. Other incident information is not enough as tickets created without all information, wrong dates, email or cui not informed. 
    2.3. Problems accesing portal because forgetten passwords, expired passwords, or url is not able.
    2.4. Configuration errors, because incorrect roles, problem with permisos, or problem with integrated systems.
    2.5. Problem with Documentation or guide, like navigation guide, or files format.
    2.6. Onboarding state in Webuy, for example supplier remais in "Onboarding". 
    2.7. Non validated for "Compra Delegada". 
    2.8. SAP propagation issues for the supplier (e.g. supplier exists in Webuy but not in SAP, invoices cannot be validated because the supplier is not registered in SAP).
    2.9. Email or user conflicts caused by previous or archived self-registrations (e.g. email already used, need to release or delete an archived user/CUI to complete a new registration).
    2.10. Other kind of problems not related to the previous categories.

#FIELD RULES (STRICT)
- Language: All JSON string values must be in Spanish.
- Create three new fields: classification, rootcause and solution.

#CLASSIFICATION FIELD
- Create a new field called *classification* with the origin of the root cause.
- The classification field can only have the following values: Group1, Group2, Group3, Group4, Group5, Group6, Group7, Group8, Group9, Group10.
- If multiple classification could apply, choose the single most specific using the following value and order:
    -- Match the value Group1 when the issue is related to not enough incident description or incident description is not provided.
    -- Match the value Group2 when the issue is related to other incident information is not enough as tickets created without all information, wrong dates, email or cui not informed. 
    -- Match the value Group3 when the issue is related to problems accesing portal because forgetten passwords, expired passwords, or url is not able.
    -- Match the value Group4 when the issue is related to configuration errors, because incorrect roles, problem with permisos, or problem with integrated systems.
    -- Match the value Group5 when the issue is related to problem with Documentation or guide, like navigation guide, or files format.
    -- Match the value Group6 when the issue is related to onboarding state in Webuy, for example supplier remais in "Onboarding". nboarding state in Webuy, for example supplier remais in "Onboarding". 
    -- Match the value Group7 when the issue is related to non validated for "Compra Delegada". 
    -- Match the value Group8 when the issue is related to SAP propagation issues for the supplier (e.g. supplier exists in Webuy but not in SAP, invoices cannot be validated because the supplier is not registered in SAP).
    -- Match the value Group9 when the issue is related to email or user conflicts caused by previous or archived self-registrations (e.g. email already used, need to release or delete an archived user/CUI to complete a new registration).
    -- Match the value Group10 when the issue is related to other kind of problems not related to the previous categories.

#ROOTCAUSE FIELD
- Create a new field called *rootcause* with the short description of the root cause.
- The rootcause field should have a max of 10 words with a concise summary of the cause. 

#SOLUTION FIELD
- Create a new filed called *solution* with a short description of the solution for the incident. 
- The solution field should have a specific, practical action the analyst would take or request. 

#QUALITY CHECK (before finalizing)
- Ensure all new fields are written in Spanish.
- Ensure the 3 fields are present and in the exact order.
- Ensure rootcause has 10 words or fewer.

#INPUT
The incident description is: '''{desc}'''
The resolution notes are: '''{res}'''
    
#FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
{{"classification": "...","rootcause": "...","solution": "..."}}
"""

In [18]:
#######################
#######################

#prompts genéricos

In [19]:
generico_prompt_consolida_analisis_general=""" 
#CONTEXT.
You are a top-tier incident analyst focused on reducing the number of tickets. 
You are contracted to analyze the main root cause of each incident and propose a concrete fix.
Recibirás como entrada una lista de análisis realizada sobre algunos incidentes.
Cada uno de estos análisis tiene el siguente formato:
**a. número de incidentes.**  
 <se incluye el número total de incidencias>

**b. análisis de incidentes.**  
**b.1 análisis detallado por causa raíz:**  
<se incluye aquí una tabla que contiene las diferentes causas raíces>

**b.2 estadísticas de causas raíz:**  
<se incluye aquí una tabla que contiene las causas raices, el número de incidentes y el porcentaje sobre el total>.

**b.3 propuestas de solución por problema común:**  
<se incluye aquí un listado de soluciones por cada causa raiz>

**c. lista de problemas raíz.**  
<se incluye aquí un listado de problemas con la relación de incidencias que se asociaan a estos problemas>

**d. conclusiones.**  
<se incluye aquí unas conclusiones interesantes y necesarias para nuestro jefe>

#TASK.
1. Consolidarás en un único informe todos los informes que se han realizado. 
2. En la consolidación, tendrás en cuenta
    2.1. El apartado **a. número de incidentes.** lo calcularás sumando el número total de incidencias de cada informe. 
    2.2. El apartado **b.1 análisis detallado por causa raíz:** lo construirás incluyendo todas las diferentes causas raices que existan en todos los informes. 
    - Si la misma causa raiz se encuentra en más de un informe, obtendrás el total de incidentes asociados a esa causa raiz sumando el número de cada informe.
    - Identificarás la misma causa raíz con un análisis semántico de su descripción sin tener en cuenta que el texto sea literalmente lo mismo.
    2.3. El apartado **b.2 estadísticas de causas raíz:** lo construirás incluyendo todas las difernetes casusass raices que existan
    - El porcentaje lo recalcularás con el total de incidentes que has usado en el punto 2.1 y el total de incidentes de causa raiz que has calculado en el punto 2.2.
    2.4. El apartado **b.3 propuestas de solución por problema común:** lo calcularás incluyendo todas las propuestas únicas que existan en todos los informes. 
    - Identificarás las propuestas por análisis semántico.
    2.5. El apartado **c. lista de problemas raíz.** lo construirás con todos los problemas distintos identificados en todos los informes. 
    2.6. El apartado **d. conclusiones.**  lo construirás con todas las recomendaciones distintas. Identificarás las mismas conclusiones según el significado semántico.

#QUALITY CHECK (before finalizing)
- Ensure the new report is done in Spanish.
- Ensure the new report has the same strucure as described in the context.
- Ensure the sum of the total incidents and the total incidents per root cause are exact and are correct.

#INPUT
<input>{json_output}</input>

"""

In [20]:
generico_prompt_analiza_todas_incidencias=""" 
    #ROLE
    You are a top-class incident analyst, focused on reducing the number of tickets. 
    You have been hired to identify the main root cause of incidents and propose viable solutions that address these causes. 
    The user will provide you with a list of incidents in JSON format.
    The name of the application is {app}

    #INPUT FORMAT (JSON)
    The list is an array of objects with the following fields:
    - id: incident id
	- desc: problem description.
	- res: resolution notes for the ticket.

    Example input (referred later as incident_list_json:
    [{{"id":"id1", "desc":"Description...","res":"Notes..."}},{{"id":"id2", "desc":"Description...","res":"Notes..."}}]
    
    #TASK
	1. Identify the main root cause for each incident.
	2. From these root causes, group common problems and propose viable solutions for each common problem.
	3. Prepare a list of common problems that includes the list of incident IDs for each problem.
	4. Respond only in the exact structure described below (A–D), using a professional tone, and in Spanish.

    #ANALYSIS RULES
	- If an incident seems to have multiple causes, choose only one main root cause (the most decisive).
	- If information is missing, infer the best possible root cause and mark it as “low confidence” when applicable.
	- Normalize wording to group similar causes (e.g., “VPN down”, “VPN error”, “VPN connectivity failure” ⇒ one single problem).
	- Include all incidents provided in the analysis and in the counts.
	- Calculate statistics for root causes (percentage of total incidents). Round to one decimal place.
	- Do not invent data. If some fields (desc or res) are missing, state it and proceed with what is available.

    #REQUIRED OUTPUT FORMAT
    Respond only with the following sections, in order, without any extra text:
    
    A. NUMBER OF INCIDENTS.
	- State the total number of incidents received.
    
    B. INCIDENT ANALYSIS.
	- B.1 Detailed analysis for each root cause: main root cause, and short justification (mention signals from desc/res).
	- B.2 Root cause statistics: list table showing each cause and its percentage of total incidents. At the end, create a table with 4 fields: root cause, short description of the root cause, the number of incidents and the porcentage that represents this number of incidents.
	- B.3 Proposed solutions for each common problem: for each problem/root cause, provide specific and viable actions (e.g., fixes, prevention, automation, monitoring, documentation, training, process changes).

    C. ROOT CAUSE LIST.
	- Return only a JSON array with this schema for each item:
	- "problema": normalized name of the common problem/root cause.
	- "incidencias": array with the id values of the incidents belonging to that problem.
    
    Example (for reference):
    [ {{"problema":"Error de conectividad con VPN","incidencias":["cod1","cod2","cod3"]}}, 
      {{"problema":"Credenciales inválidas SSO","incidencias":["cod4"]}}
    ]

    D. CONCLUSIONS.
	• Clear, actionable conclusions (e.g., quick wins, risks, priorities, follow-up metrics).
    
    E. VERIFICATION BEFORE ANSWERING
	- Make sure all incidents from incident_list_json are present in sections A–C.
	- Recount the total number and confirm it matches the value in section A.
	- Ensure the final answer is entirely in Spanish; if not, translate it into Spanish before returning it.
    - Verify the number of inciddents


    #INPUT / inciden_list_json
    The incident list is: {listado_json}
    The number of incidents in the list should be {numero_incidentes}
    """

In [21]:
generico_genera_propuestas=""" 
#CONTEXTO.
    Hay un sistema, llamado Tell Me, que es una arquitectura de RAG, integrado en el proceso de apertura de 
    tickets. En Tell Me se ingesta documentación para guiar al usuario en el proceso de apertura, de forma que se pueda reducir el número de tickets. En la actualidad tenemos un 30% de reducción
    de apertura de tickets.

    Tenemos un listado de incidencias de la aplicación {APP} que han sido analizados, generándose los campos:
    * root_cause: Es la causa raíz de la incidencia. 
    * solution: Es el resumen de la solución que se ha dado al usuario. 

    Te han contratado con el rol es de analista de procesos, tienes en el contrato una prima de 2000$ si logras 
    aumentar la reducción desde el 30% al 60%. Si no haces un buen trabajo, a mi me despedirán y tú no tendrás ningún 
    pago. 

    #FORMATO DE ENTRADA (JSON)
    Lista de objetos con 
	• root_cause: causa raíz de la incidencia.
	• solution: resumen de la solución del ticket.
	
    Example input (referred later as incident_list_json:
    [{{"root_cause":"...","solution":"..."}},{{"root_cause":"...","solution":"..."}}]
    
    #TAREA
	* Agrupa cada causa raíz por simulitud semántico.
	* Extrae de las solution la información exacta y necesaria que un chatbot debe entregar para que el usuario resuelva cada tipo de incidencia. 
    * Genera la documentación para el sistema RAG, organizada por los grupos de causas raíz.
    * Propón mejoras en los sistemas que reduzcan la recurrencia de estos problemas.

    #INSTRUCCIONES
    * Usa exclusivamente la información del input. No inventes datos. 
    * Elimina duplicidades y evita contradicciones.
    * La agrupación debe hacerse solo por significado operativo, no por similitud superficial de palabras.
    * Asigna a cada grupo un nombre corto y consistente, y úsalo en toda la salida.
    * Las propuestas de mejora deben estar claramente vinculadas a los grupos de causas raíz afectados.
    * Responde siempre en español con redacción clara y útil para su incorporación a un RAG.
    

    #FORMATO DE SALIDA REQUERIDO
    1. **NÚMERO DE INCIDENCIAS**
	* Indica cuántos elementos contiene la lista y explica brevemente el método de conteo.

	2.	**DOCUMENTACIÓN PARA RAG**
	* Presenta la información consolidada y sin duplicados, organizada por cada grupo de causas raíz (usar el nombre del grupo como encabezado).
	* Para cada grupo:
	    - Resumen de la causa raíz.
	    - Información que el chatbot debe comunicar al usuario para resolverla, basada en las solution originales.
    * Usa este formato para la documentación:
        - Usa un tono formal, amigable, y directo. 
        - Desarrolla el texto manteniendo una introducción seguida de una serie de acciones a realizar. 
        - Tanto la parte de introducción como cada una de las reglas debe tener una longitud de al menos 3 líneas. 

	3. **PROPUESTAS DE MODIFICACIÓN EN SISTEMAS**
	* Cambios necesarios en los sistemas origen.
	* Cada cambio debe indicar explícitamente a qué grupo de causa raíz beneficia.
    
	4. **CONCLUSIONES**
	* Hallazgos clave y recomendaciones para la reducción de incidencias.
    * Deben ser conclusiones útiles para tu jefe y para el usuario.
    
    #VERIFICACIÓN
    * Validar que toda la salida es veraz respecto al input.
    * Verificar ausencia de incoherencias o información contradictoria.
    * Confirmar que la salida está en español.
    * Asegurar que las conclusiones explican claramente qué está causando el volumen de incidencias.
    * En concreto el listado tiene {len_elementos} elementos. Verifica que cuadra con la información que has incluido. 
    
    #INPUT
    {listado_json}
"""

In [22]:
generico_genera_promtp=""" 
#Contexto
Eres un ingeniero de prompts y ayudas a la generación de prompts automáticos en el análisis de incidencias. 
Trabajas en cadena con dos colegas que son expertos en el análisis de incidencias. A partir del trabajo que realiza el primer colega, y a partir de un prompt genérico para otra aplicación, te encargas de generar el mejor prompt posible para la nueva aplicación. 
Si haces bien tu trabajo te darán una prima de 2000$. En cambio, si tu trabajo no es apropiado y se rompe la cadena de análisis, te despedirán a tí y a mi por contarte.

#Tarea
A partir de un prompt, incluido entre las etiquetas de xml <prompt_generico>, y de un análisis de incidencias donde se recogen las causas raíces de las mismas, incluido en las etiquetas xml <analisis_especifico>, te encargarás de devolver un prompt adecuado para el análisis incidencia a incidencia para esa aplicación. 

El prompt genérico tiene distintas partes, entre ellas un apartado TASK y un apartado CLASSIFICATION FIELD, que son los que tendrás que modificar para actualizar con respecto al análisis de incidencias que se ha generado.

#Instrucciones
Modificarás el apartado 2 dentro del título TASK para contemplar la agrupación que te da el analista de incidencias. 
Modificarás el apartado 2 dentro del título CLASSIFICATION FIELD para contabilizar sólo el número de grupos máximos que ha considerado el analista de incidencias. 
Modificarás el apartado 3 dentro del título CLASSIFICATION FIELD para especificar las reglas de macheo del nombre de los grupos con el análisis de causa raíz de las mismas. 
Todas las modificaciones las realizarás en inglés.
Si el prompt que te inyecto tiene un apartado de <thinking> lo revisarás por si hay algún aspecto que no haya sido incluido en la respuesta final.

#Especificación
Sólo responderás como salida el prompt. 
Sólo responderás en el lenguaje inglés. 
No responderás sólo un json.

#Validación
Validarás que tu respuesta se corresponde con el prompt genérico, pero actualizando la información del analista.
Validarás que todos los subapartados del apartado 2 del título TASK tienen el mismo número de elementos que el apartado 2 del título CLASSIFICATION FIELD.
Validarás que el apartado 2 y 3 del título CLASSIFICATION FIELD tienen el mismo número de grupos. 
Validarás que el resto del prompt no se ha modificado. 
Validarás que la salida que hará

#PROMPT GENERICO
<prompt_generico>
#CONTEXT.
    You are a top-tier incident analyst focused on reducing the number of tickets. 
    You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
    Each incident provides three fields:
    • id: incident code.
    • desc: problem description.
    • res: resolution notes for the ticket.

#TASK.
    1. From one incident, identify the main problem(s) and propose a viable, actionable solution.

    2. Based on the incident description and resolution, detect which of the following groups best matches the main root cause. 
       Only assign a group when it is explicitly supported by the text:
       2.1. Supplier self-registered in Webuy instead of being registered via the Enel contact (gestor) in the “compra delegada” process.
       2.2. Wrong registration status or onboarding state in Webuy (e.g. supplier remains in “Onboarding”, not validated for “compra delegada”, or data not migrated to the correct CUI/SAP record).
       2.3. SAP propagation issues for the supplier (e.g. supplier exists in Webuy but not in SAP, invoices cannot be validated because the supplier is not registered in SAP).
       2.4. Email or user conflicts caused by previous or archived self-registrations (e.g. email already used, need to release or delete an archived user/CUI to complete a new registration).
       2.5. PDA or economic/financial data not updated or not calculated correctly due to incorrect registration workflow or status.
       2.6. Other process or access issues related to the Webuy–SAP registration workflow (including DRAPE access problems derived from incorrect registration or onboarding).
       2.7. Other kind of problems not related to the previous categories.

#FIELD RULES (STRICT)
- Language: All JSON string values must be in Spanish.
- Create three new fields: classification, rootcause and solution.

#CLASSIFICATION FIELD
- Create a new field called *classification* with the origin of the root cause.
- The classification field can only have the following values: Group1, Group2, Group3, Group4, Group5, Group6, Group7.
- If multiple classification could apply, choose the single most specific using the following value and order:
    -- Match the value Group1 when the issue is supplier self-registration in Webuy instead of the correct “compra delegada” process through the Enel contact (gestor).
    -- Match the value Group2 when the issue is an incorrect registration status or onboarding state in Webuy (e.g. still “Onboarding”, not validated for “compra delegada”, or not migrated correctly).
    -- Match the value Group3 when the issue is SAP propagation (supplier not created or updated in SAP, or invoices blocked because the supplier is not registered in SAP).
    -- Match the value Group4 when the issue is an email or user conflict due to previous or archived self-registrations (e.g. email already used by an archived account).
    -- Match the value Group5 when the issue is PDA or economic/financial data that is not updated or not calculated because of registration/status problems.
    -- Match the value Group6 when the issue is another process or access problem related to the Webuy–SAP registration workflow (including DRAPE access) derived from the registration process misunderstanding.
    -- Match the value Group7 when the issue is any other problem not covered by the previous groups.

#ROOTCAUSE FIELD
- Create a new field called *rootcause* with the short description of the root cause.
- The rootcause field should have a max of 10 words with a concise summary of the cause. 

#SOLUTION FIELD
- Create a new filed called *solution* with a short description of the solution for the incident. 
- The solution field should have a specific, practical action the analyst would take or request. 

#QUALITY CHECK (before finalizing)
- Ensure all new fields are written in Spanish.
- Ensure the 3 fields are present and in the exact order.
- Ensure rootcause has 10 words or fewer.

#INPUT
The incident description is: '{desc}'
The resolution notes are: '{res}'
    
#FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
{{"classification": "...","rootcause": "...","solution": "..."}}
</prompt_generico>

#ANALISIS ESPECIFICO
<analisis_especifico>
{analisis_incidencias}
</analisis_especifico>
"""

In [23]:
generico_genera_descripcion_solucion=""" 
#ROLE
You are a top-class incident analyst, focused on reducing the number of tickets.
You have been hired to normalize an incident description and the incident resolution
of a given ticket.
The name of the application is {app_name}

#INPUT DATA
The following fields are used to calculate the outputs.
* id: ticket code.
* descripcion: problem description. Sometimes blank as it is autogenerated.
* descripcion_corta: short description.
* nota_resolucion: resolution notes. Sometimes blank as the ticket is not closed yet.
* nota_trabajo: working notes.
* comentario_nota_trabajo: more working notes (optional).
* estado: status of the ticket.

#TASK
You must produce FIVE fields:
1) desc: short description of the problem/request (only what happens is needed).
2) res: short description of the action/solution.
3) rootcause: canonical root cause label (max 10 words).
4) res_type: Real | Estimado
5) ticket_type: Incidencia | Peticion | Indeterminado

#FIELD RULES (STRICT)
- Language: All JSON string values must be in Spanish.
- Create five new fields: desc, res, rootcause, res_type, ticket_type.
- Output JSON must include the 5 fields in the exact order:
  {{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}

#INTERNAL ANALYSIS STEPS (MANDATORY)
Before generating the final JSON, do these steps internally (do NOT output them):
A) Extract:
   A1) OBBSERVED FACTS: what is wrong / what fails / what impact is there?
   A2) REQUESTED ACTION: What is the user or the operator asking for?
B) Decide ticket_type using STRICT rules (see DISAMBIGUATION).
C) Generate desc/res/rootcause/res_type consistent with ticket_type.

#DISAMBIGUATION (STRICT)  <-- THIS IS CRITICAL
You MUST NOT classify based on keywords alone (e.g., "modificar", "actualizar", "cambiar").

## Assign ticket_type = Incidencia when ANY of these is true:
- There are signals of malfunction/defect/impact, e.g.:
  "mal", "incorrecto", "erróneo", "incompleto", "no corresponde", "descuadre",
  "se grabó mal", "no se actualiza", "error", "fallo", "no funciona", "bloquea",
  "inconsistencia", "impacta", "afecta", "no permite", "no carga", "no valida".
- The request is to modify/update data BECAUSE the data is wrong or the system registered it incorrectly
  (e.g., "corregir", "rectificar", "subsanar", "revertir", "dato mal informado").
- There is mention of a technical/process defect cause (bug, integración, migración, carga, parametrización).

## Assign ticket_type = Peticion when ALL of these are true:
- No signals of malfunction/defect/impact (no "mal/incorrecto/error/fallo/bloqueo" etc.).
- The text is a service request or consultation, such as:
  - A status check (“status?”, “follow-up”, “when will it be done?”, “information”).
  - A voluntary request for change/onboarding/offboarding/configuration/access/permissions WITHOUT indicating something is wrong.

## Assign ticket_type = Indeterminado when:
- There is insufficient information and you cannot infer whether there is a defect/impact or a voluntary request.

#DESC FIELD
- Create *desc* as a short and concise summary of what happens or is needed.
- desc must NOT include solution nor root cause, only the situation.
- If ticket_type = Peticion, desc should reflect the request/consultation.
- If ticket_type = Incidencia, desc should reflect the problem/impact.

#RES FIELD
- Create *res* with a short description of the solution/action taken.
- If ticket_type = Peticion:
  - If it is only a consultation (status/info) and no manual action is required:
    res = "Este ticket es una petición, no una incidencia."
  - If it is a request that requires a manual action (e.g., alta/baja/cambio de permisos/configuración):
    res = "Este ticket es una petición, no una incidencia. Se realizó la gestión solicitada."
    (If no evidence of completion exists, omit the second sentence.)
- If ticket_type = Incidencia:
  - res must describe the real practical action taken to resolve it, if present.
  - If no clear solution is contained but it is clear that a manual action was done by the operator:
    res = "Esta incidencia no tiene una solución clara especificada pero se ha realizado una acción manual por el operador."
  - Otherwise, if there is no explicit resolution:
    res = "Esta incidencia no tiene una solución clara especificada en la información."

#ROOTCAUSE FIELD
- Create *rootcause* with max 10 words.
- Use canonical labels: normalize equivalent meanings to the same short label.
- Do NOT invent causes.

## Rootcause rules by ticket_type:
- If ticket_type = Peticion:
  - If it is a status/info request: rootcause = "Consulta de estado o información."
  - If it is a service request (alta/baja/cambio/acceso): rootcause = "Solicitud de servicio sin fallo."
- If ticket_type = Incidencia:
  - If the ticket is closed as duplicated: rootcause = "Incidencia duplicada."
  - If you cannot understand rootcause but manual action is clear:
    rootcause = "Sin rootcause clara; acción manual permitió continuar."
  - If you cannot understand rootcause:
    rootcause = "Sin rootcause clara definida en la información."

#RES_TYPE FIELD
- res_type can only be: Real, Estimado
- Use Real when:
  - ticket_type = Incidencia AND there is a clear action/solution done.
  - OR ticket_type = Peticion AND there is evidence that the requested management was performed.
- Use Estimado in all other cases.

#FEW-SHOT EXAMPLES (IMPORTANT)

## Example 1: INCIDENCIA (modificar dato por error)
Input:
"Modificar CIF del cliente: está mal informado y afecta facturas."
Output:
{{"desc":"CIF de cliente incorrecto y afecta facturas.","res":"Se corrigió el CIF en el maestro de clientes.","rootcause":"Dato maestro incorrecto.","res_type":"Real","ticket_type":"Incidencia"}}

## Example 2: PETICIÓN (cambio voluntario sin fallo)
Input:
"Solicito cambiar el CIF del cliente por cambio societario. No hay errores."
Output:
{{"desc":"Solicitud de cambio de CIF por cambio societario.","res":"Este ticket es una petición, no una incidencia.","rootcause":"Solicitud de servicio sin fallo.","res_type":"Estimado","ticket_type":"Peticion"}}

## Example 3: INCIDENCIA (fallo funcional)
Input:
"El sistema no actualiza el domicilio aunque lo guardo."
Output:
{{"desc":"El domicilio no se actualiza al guardarlo.","res":"Se aplicó corrección para permitir la actualización del domicilio.","rootcause":"Fallo en actualización de datos.","res_type":"Real","ticket_type":"Incidencia"}}

## Example 4: PETICIÓN (consulta de estado)
Input:
"¿Estado del ticket 123? ¿Cuándo estará resuelto?"
Output:
{{"desc":"Consulta del estado del ticket 123.","res":"Este ticket es una petición, no una incidencia.","rootcause":"Consulta de estado o información.","res_type":"Estimado","ticket_type":"Peticion"}}

## Example 5: INCIDENCIA (integración)
Input:
"Se cargó mal el IBAN por integración."
Output:
{{"desc":"IBAN cargado incorrectamente por la integración.","res":"Se corrigió el IBAN y se revisó la integración.","rootcause":"Error de integración en carga de datos.","res_type":"Real","ticket_type":"Incidencia"}}

## Example 6: DUPLICADA
Input:
"Cerrar, duplicada de INC-999."
Output:
{{"desc":"Solicitud de cierre por incidencia duplicada.","res":"Se cerró el ticket por duplicidad.","rootcause":"Incidencia duplicada.","res_type":"Real","ticket_type":"Incidencia"}}

#QUALITY CHECK (before finalizing)
- Ensure all new fields are in Spanish.
- Ensure the 5 fields are present and in the exact order.
- Ensure rootcause has max 10 words.
- Ensure ticket_type matches DISAMBIGUATION rules.

#INPUT
<id>{id}</id>
<descripcion>{description}</descripcion>
<descripcion_corta>{descripcion_corta}</descripcion_corta>
<nota_resolucion>{nota_resolucion}</nota_resolucion>
<nota_trabajo>{nota_trabajo}</nota_trabajo>
<comentario_nota_trabajo>{comentario_nota_trabajo}</comentario_nota_trabajo>
<estado>{estado}</estado>

#FINAL OUTPUT FORMAT (return only this JSON)
{{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}
"""

pp="""
#ROLE
    You are a top-class incident analyst, focused on reducing the number of tickets.
    You have been hired to normalize an incident description and the incident resolution 
    of a given incident. 
    The name of the application is {app_name}

    #INPUT DATA
    The following fields are used to calculate the descripcion and the resolution. 
	* id: incident code.
	* descripcion: problem description. Sometimes is blank as it is autogenerated. 
    * descripcion_corta: short problem description. 
	* nota_resolucion: resolution notes. Sometimes is blank as the incident is not closed yet. 
    * nota_trabajo: working notes. 
    * comentario_nota_trabajo: more workings notes. This field is optional so it is posible it is not in the call.
    * estado: status of the incident. May it be resolved or may it be yet pending.
 
    #TASK
	1. Identify the problem description of the incident. 
    2. Identify the problem resolution of the incident. 
    3. Identify the root cause of the incident.
    4. Identify the resolution type. It can be only two values:
        4.1. Real.
        4.2. Estimado.
    
    #FIELD RULES (STRICT)
    - Language: All JSON string values must be in Spanish.
    - Create four new fields: desc, res, rootcause and res_type.

    #DESC FIELD
    - Create a new field called *desc* with a short description included in the incident.
    - The description field should be a short and concise summary of the description of the incident. 
    - The description field does not included anything regarding the solution or the rootcause. Only the description of the problem.

    #RES FIELD
    - Create a new filed called *res* with a short description of the solution for the incident. 
    - The solution field should have a specific, practical action taken to resolve the incident. 
    - The solution field must have the real resolution done by the incident. 
    - If the information is not about a problem, but it is a request that not requires a manual update, for example to know the status of a ticket or process, then must included as solution: `Este ticket es una petición, no una incidencia.`
    - If there is not a clear solution contained in the information, but it is clear that has been make some manual action by the operator must included as solution: `Esta incidencia no tiene una solución clara especificada pero se ha realizado una acción manual por el operador.`
    - In other case, if the incident does not have exactly a resolution (for example requesting a later call or dosn't have an explicit solution) and it doesn't include a manual update from the operator, must included as solution: `Esta incidencia no tiene una solución clara especificada en la información.`

    TODO --> Meter incidencias duplicadas, meter peticiones que no son incidencias, como preguntar por el estado de la petición.

    #ROOTCAUSE FIELD
    - Create a new field called *rootcause* with the short description of the root cause.
    - The rootcause field should have a max of 10 words with a concise summary of the cause. 
    - Generate the rootcause with semmantic meaning, creating canonic labels (short sentences) for some equivalents meaning.
    - Don't invent the root cause, just identify the main root cause contained in the information.
    - If the information is not about a problem, but it is a request but it is a request that not requires a manual update, for example to know the status of a ticket or a process, then must included as rootcause: `Falta documentación para entender el flujo del proceso.` 
    - If the information is about closing the ticket because it is duplicated, then must included as rootcause: `Incidencia duplicada.`
    - If you can not undestando the rootcause but it is clear that some manual update is done by the operator, must included as rootcouse: `Esta incidencia no tiene una rootcause clara especificada pero una acción manual permite seguir el proceso.`
    - If you can not understand the root cause must included as rootcause: `Esta incidencia no tiene una rootcause clara definida en la información.`
    
    #RES_TYPE FIELD
    - Create a new filed called *res_type* with the type of resolution.
    - The classification field can only have the following values: Real, Estimado.
        -- Match the value Real when the incident information has a clear solution done.
        -- Match the value Estimado in other cases.

    #QUALITY CHECK (before finalizing)
    - Ensure all new fields are written in Spanish.
    - Ensure the 4 fields are present and in the exact order. 
    - Ensure the rootcause is present.
    
    #INPUT
    <id>{id}</id>
    <descripcion>{description}</descripcion>
    <descripcion_corta>{descripcion_corta}</descripcion_corta>
    <nota_resolucion>{nota_resolucion}</nota_resolucion>
    <nota_trabajo>{nota_trabajo}</nota_trabajo>
    <comentario_nota_trabajo>{comentario_nota_trabajo}</comentario_nota_trabajo>
    <estado>{estado}</estado>
    
    #FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
    {{"desc": "...", "res": "...", "rootcause":"...", "res_type": "..."}}
"""

In [24]:
generico_genera_descripcion_solucion

' \n#ROLE\nYou are a top-class incident analyst, focused on reducing the number of tickets.\nYou have been hired to normalize an incident description and the incident resolution\nof a given ticket.\nThe name of the application is {app_name}\n\n#INPUT DATA\nThe following fields are used to calculate the outputs.\n* id: ticket code.\n* descripcion: problem description. Sometimes blank as it is autogenerated.\n* descripcion_corta: short description.\n* nota_resolucion: resolution notes. Sometimes blank as the ticket is not closed yet.\n* nota_trabajo: working notes.\n* comentario_nota_trabajo: more working notes (optional).\n* estado: status of the ticket.\n\n#TASK\nYou must produce FIVE fields:\n1) desc: short description of the problem/request (only what happens is needed).\n2) res: short description of the action/solution.\n3) rootcause: canonical root cause label (max 10 words).\n4) res_type: Real | Estimado\n5) ticket_type: Incidencia | Peticion | Indeterminado\n\n#FIELD RULES (STRIC

In [25]:

data={
            "OnBoarding": {
                "analiza_incidencia_causa_raiz": prompt_onboarding_causa_raiz
            },
            "HrAnalytics": {
                "analiza_incidencia_causa_raiz": prompt_hranalytics_causa_raiz
            },
            "WeBuy": {
                "analiza_incidencia_causa_raiz": prompt_webuy_causa_raiz
            },
            "generico": {
                "analiza_todas_incidencias": generico_prompt_analiza_todas_incidencias,
                "genera_propuestas": generico_genera_propuestas,
                "genera_prompt_app": generico_genera_promtp,
                "calcula_descripcion_solucion": generico_genera_descripcion_solucion,
                "consolida_analisis_general" : generico_prompt_consolida_analisis_general
            }
        }

In [26]:
import json

with open("prompts.ini", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
    